####  Configure ADLS Connection

In [0]:
storage_account_name = "stsupplychainlak"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    dbutils.secrets.get(
        scope="adls-scope",
        key="storage-account-key"
    )
)

#### Read Bronze Tables


In [0]:
yellow_taxi_df = spark.read.format("delta").load(
    "abfss://bronze@stsupplychainlak.dfs.core.windows.net/yellow_taxi"
)

zone_lookup_df = spark.read.format("delta").load(
    "abfss://bronze@stsupplychainlak.dfs.core.windows.net/taxi_zone_lookup"
)

display(yellow_taxi_df.limit(25))
display(zone_lookup_df.limit(25))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-07-01T00:29:37,2025-07-01T00:45:30,1,7.3,1,N,138,74,1,29.6,7.75,0.5,9.0,6.94,1.0,54.79,0.0,1.75,0.0
1,2025-07-01T00:23:28,2025-07-01T01:07:44,1,17.7,2,N,132,142,1,70.0,4.25,0.5,5.0,0.0,1.0,80.75,2.5,1.75,0.0
2,2025-07-01T00:53:50,2025-07-01T01:27:12,1,9.98,1,N,138,48,1,43.6,6.0,0.5,10.87,0.0,1.0,66.97,2.5,1.75,0.75
2,2025-07-01T00:58:49,2025-07-01T01:15:55,1,10.27,1,N,138,229,1,38.7,6.0,0.5,14.1,6.94,1.0,72.24,2.5,1.75,0.75
2,2025-07-01T00:09:22,2025-07-01T00:23:54,1,2.94,1,N,211,97,1,17.0,1.0,0.5,3.0,0.0,1.0,25.75,2.5,0.0,0.75
1,2025-07-01T00:39:14,2025-07-01T00:55:21,1,11.8,1,N,132,155,1,44.3,1.0,0.5,14.05,0.0,1.0,60.85,0.0,0.0,0.0
2,2025-07-01T00:15:26,2025-07-01T00:29:39,1,3.87,1,N,79,263,1,17.7,1.0,0.5,4.69,0.0,1.0,28.14,2.5,0.0,0.75
2,2025-07-01T00:40:58,2025-07-01T00:44:15,1,0.85,1,N,140,262,1,5.8,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,0.0
2,2025-07-01T00:28:12,2025-07-01T00:39:49,2,2.54,1,N,114,50,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75
2,2025-07-01T00:38:17,2025-07-01T00:55:44,1,6.37,1,N,132,197,1,26.8,1.0,0.5,5.86,0.0,1.0,36.91,0.0,1.75,0.0


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


#### Explore Data

In [0]:
print("Total Rows :", yellow_taxi_df.count())
print("Total Columns :", len(yellow_taxi_df.columns))

yellow_taxi_df.printSchema()

Total Rows : 39547664
Total Columns : 20
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [0]:
print("Total Rows :", zone_lookup_df.count())
print("Total Columns :", len(zone_lookup_df.columns))

zone_lookup_df.printSchema()

Total Rows : 265
Total Columns : 4
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [0]:
from pyspark.sql.functions import *
display(
    yellow_taxi_df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in yellow_taxi_df.columns
    ])
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8638094494045474>, line 3
      1 from pyspark.sql.functions import *
      2 display(
----> 3     yellow_taxi_df.select([
      4         count(
      5             when(col(c).isNull(), c)
      6         ).alias(c)
      7         for c in yellow_taxi_df.columns
      8     ])
      9 )

NameError: name 'yellow_taxi_df' is not defined

#### Check Invalid Records

In [0]:
from pyspark.sql.functions import *

print(
    "Passenger Count <= 0 :",
    yellow_taxi_df.filter(
        col("passenger_count") <= 0
    ).count()
)

print(
    "Trip Distance <= 0 :",
    yellow_taxi_df.filter(
        col("trip_distance") <= 0
    ).count()
)

print(
    "Fare Amount <= 0 :",
    yellow_taxi_df.filter(
        col("fare_amount") <= 0
    ).count()
)

print(
    "Total Amount <= 0 :",
    yellow_taxi_df.filter(
        col("total_amount") <= 0
    ).count()
)

Passenger Count <= 0 : 173411
Trip Distance <= 0 : 1204829
Fare Amount <= 0 : 1648765
Total Amount <= 0 : 636329


In [0]:
silver_df = yellow_taxi_df.filter(
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    (col("total_amount") > 0)
)

In [0]:
print("Rows Before Cleaning :", yellow_taxi_df.count())
print("Rows After Cleaning  :", silver_df.count())

Rows Before Cleaning : 39547664
Rows After Cleaning  : 36834546


#### Feature Engineering

In [0]:
silver_df = silver_df \
.withColumn(
    "pickup_year",
    year(col("tpep_pickup_datetime"))
) \
.withColumn(
    "pickup_month",
    month(col("tpep_pickup_datetime"))
) \
.withColumn(
    "pickup_day",
    dayofmonth(col("tpep_pickup_datetime"))
) \
.withColumn(
    "pickup_hour",
    hour(col("tpep_pickup_datetime"))
) \
.withColumn(
    "trip_duration_minutes",
    round(
        (
            unix_timestamp(col("tpep_dropoff_datetime"))
            -
            unix_timestamp(col("tpep_pickup_datetime"))
        ) / 60,
        2
    )
)

In [0]:
display(
    silver_df.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "pickup_year",
        "pickup_month",
        "pickup_day",
        "pickup_hour",
        "trip_duration_minutes"
    ).limit(25)
)

tpep_pickup_datetime,tpep_dropoff_datetime,pickup_year,pickup_month,pickup_day,pickup_hour,trip_duration_minutes
2025-07-01T00:29:37,2025-07-01T00:45:30,2025,7,1,0,15.88
2025-07-01T00:23:28,2025-07-01T01:07:44,2025,7,1,0,44.27
2025-07-01T00:53:50,2025-07-01T01:27:12,2025,7,1,0,33.37
2025-07-01T00:58:49,2025-07-01T01:15:55,2025,7,1,0,17.1
2025-07-01T00:09:22,2025-07-01T00:23:54,2025,7,1,0,14.53
2025-07-01T00:39:14,2025-07-01T00:55:21,2025,7,1,0,16.12
2025-07-01T00:15:26,2025-07-01T00:29:39,2025,7,1,0,14.22
2025-07-01T00:40:58,2025-07-01T00:44:15,2025,7,1,0,3.28
2025-07-01T00:28:12,2025-07-01T00:39:49,2025,7,1,0,11.62
2025-07-01T00:38:17,2025-07-01T00:55:44,2025,7,1,0,17.45


In [0]:
silver_df.filter(
    col("trip_duration_minutes") <= 0
).count()

519845

#### Filter invalid trips with zero or negative trip duration.

In [0]:
silver_df = silver_df.filter(
    col("trip_duration_minutes") > 0
)

In [0]:
print("Final Silver Rows :", silver_df.count())

Final Silver Rows : 36314701


#### Enrich taxi trips with pickup and dropoff location details.

In [0]:
# Creating Pickup Lookup
pickup_lookup = zone_lookup_df.select(
    col("LocationID").alias("PULocationID"),
    col("Borough").alias("Pickup_Borough"),
    col("Zone").alias("Pickup_Zone"),
    col("service_zone").alias("Pickup_Service_Zone")
)

In [0]:
# Create Dropoof Lookup
dropoff_lookup = zone_lookup_df.select(
    col("LocationID").alias("DOLocationID"),
    col("Borough").alias("Dropoff_Borough"),
    col("Zone").alias("Dropoff_Zone"),
    col("service_zone").alias("Dropoff_Service_Zone")
)

In [0]:
# Join Pickup
silver_df = silver_df.join(
    pickup_lookup,
    on="PULocationID",
    how="left"
)

In [0]:
# Join Dropoff
silver_df = silver_df.join(
    dropoff_lookup,
    on="DOLocationID",
    how="left"
)

In [0]:
# Validate
display(
    silver_df.select(
        "PULocationID",
        "Pickup_Borough",
        "Pickup_Zone",
        "DOLocationID",
        "Dropoff_Borough",
        "Dropoff_Zone"
    ).limit(25)
)

PULocationID,Pickup_Borough,Pickup_Zone,DOLocationID,Dropoff_Borough,Dropoff_Zone
138,Queens,LaGuardia Airport,74,Manhattan,East Harlem North
132,Queens,JFK Airport,142,Manhattan,Lincoln Square East
138,Queens,LaGuardia Airport,48,Manhattan,Clinton East
138,Queens,LaGuardia Airport,229,Manhattan,Sutton Place/Turtle Bay North
211,Manhattan,SoHo,97,Brooklyn,Fort Greene
132,Queens,JFK Airport,155,Brooklyn,Marine Park/Mill Basin
79,Manhattan,East Village,263,Manhattan,Yorkville West
140,Manhattan,Lenox Hill East,262,Manhattan,Yorkville East
114,Manhattan,Greenwich Village South,50,Manhattan,Clinton West
132,Queens,JFK Airport,197,Queens,Richmond Hill


In [0]:
display(silver_df.limit(25))

DOLocationID,PULocationID,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_year,pickup_month,pickup_day,pickup_hour,trip_duration_minutes,Pickup_Borough,Pickup_Zone,Pickup_Service_Zone,Dropoff_Borough,Dropoff_Zone,Dropoff_Service_Zone
74,138,1,2025-07-01T00:29:37,2025-07-01T00:45:30,1,7.3,1,N,1,29.6,7.75,0.5,9.0,6.94,1.0,54.79,0.0,1.75,0.0,2025,7,1,0,15.88,Queens,LaGuardia Airport,Airports,Manhattan,East Harlem North,Boro Zone
142,132,1,2025-07-01T00:23:28,2025-07-01T01:07:44,1,17.7,2,N,1,70.0,4.25,0.5,5.0,0.0,1.0,80.75,2.5,1.75,0.0,2025,7,1,0,44.27,Queens,JFK Airport,Airports,Manhattan,Lincoln Square East,Yellow Zone
48,138,2,2025-07-01T00:53:50,2025-07-01T01:27:12,1,9.98,1,N,1,43.6,6.0,0.5,10.87,0.0,1.0,66.97,2.5,1.75,0.75,2025,7,1,0,33.37,Queens,LaGuardia Airport,Airports,Manhattan,Clinton East,Yellow Zone
229,138,2,2025-07-01T00:58:49,2025-07-01T01:15:55,1,10.27,1,N,1,38.7,6.0,0.5,14.1,6.94,1.0,72.24,2.5,1.75,0.75,2025,7,1,0,17.1,Queens,LaGuardia Airport,Airports,Manhattan,Sutton Place/Turtle Bay North,Yellow Zone
97,211,2,2025-07-01T00:09:22,2025-07-01T00:23:54,1,2.94,1,N,1,17.0,1.0,0.5,3.0,0.0,1.0,25.75,2.5,0.0,0.75,2025,7,1,0,14.53,Manhattan,SoHo,Yellow Zone,Brooklyn,Fort Greene,Boro Zone
155,132,1,2025-07-01T00:39:14,2025-07-01T00:55:21,1,11.8,1,N,1,44.3,1.0,0.5,14.05,0.0,1.0,60.85,0.0,0.0,0.0,2025,7,1,0,16.12,Queens,JFK Airport,Airports,Brooklyn,Marine Park/Mill Basin,Boro Zone
263,79,2,2025-07-01T00:15:26,2025-07-01T00:29:39,1,3.87,1,N,1,17.7,1.0,0.5,4.69,0.0,1.0,28.14,2.5,0.0,0.75,2025,7,1,0,14.22,Manhattan,East Village,Yellow Zone,Manhattan,Yorkville West,Yellow Zone
262,140,2,2025-07-01T00:40:58,2025-07-01T00:44:15,1,0.85,1,N,1,5.8,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,0.0,2025,7,1,0,3.28,Manhattan,Lenox Hill East,Yellow Zone,Manhattan,Yorkville East,Yellow Zone
50,114,2,2025-07-01T00:28:12,2025-07-01T00:39:49,2,2.54,1,N,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75,2025,7,1,0,11.62,Manhattan,Greenwich Village South,Yellow Zone,Manhattan,Clinton West,Yellow Zone
197,132,2,2025-07-01T00:38:17,2025-07-01T00:55:44,1,6.37,1,N,1,26.8,1.0,0.5,5.86,0.0,1.0,36.91,0.0,1.75,0.0,2025,7,1,0,17.45,Queens,JFK Airport,Airports,Queens,Richmond Hill,Boro Zone


#### Store the transformed and enriched dataset in the Silver layer as a Delta table.

In [0]:
# Write Silver Delta
silver_df.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://silver@stsupplychainlak.dfs.core.windows.net/yellow_taxi"
)

In [0]:
# Validate Silver Layer
silver_table = spark.read.format("delta").load(
    "abfss://silver@stsupplychainlak.dfs.core.windows.net/yellow_taxi"
)

display(silver_table.limit(25))

DOLocationID,PULocationID,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_year,pickup_month,pickup_day,pickup_hour,trip_duration_minutes,Pickup_Borough,Pickup_Zone,Pickup_Service_Zone,Dropoff_Borough,Dropoff_Zone,Dropoff_Service_Zone
74,138,1,2025-07-01T00:29:37,2025-07-01T00:45:30,1,7.3,1,N,1,29.6,7.75,0.5,9.0,6.94,1.0,54.79,0.0,1.75,0.0,2025,7,1,0,15.88,Queens,LaGuardia Airport,Airports,Manhattan,East Harlem North,Boro Zone
142,132,1,2025-07-01T00:23:28,2025-07-01T01:07:44,1,17.7,2,N,1,70.0,4.25,0.5,5.0,0.0,1.0,80.75,2.5,1.75,0.0,2025,7,1,0,44.27,Queens,JFK Airport,Airports,Manhattan,Lincoln Square East,Yellow Zone
48,138,2,2025-07-01T00:53:50,2025-07-01T01:27:12,1,9.98,1,N,1,43.6,6.0,0.5,10.87,0.0,1.0,66.97,2.5,1.75,0.75,2025,7,1,0,33.37,Queens,LaGuardia Airport,Airports,Manhattan,Clinton East,Yellow Zone
229,138,2,2025-07-01T00:58:49,2025-07-01T01:15:55,1,10.27,1,N,1,38.7,6.0,0.5,14.1,6.94,1.0,72.24,2.5,1.75,0.75,2025,7,1,0,17.1,Queens,LaGuardia Airport,Airports,Manhattan,Sutton Place/Turtle Bay North,Yellow Zone
97,211,2,2025-07-01T00:09:22,2025-07-01T00:23:54,1,2.94,1,N,1,17.0,1.0,0.5,3.0,0.0,1.0,25.75,2.5,0.0,0.75,2025,7,1,0,14.53,Manhattan,SoHo,Yellow Zone,Brooklyn,Fort Greene,Boro Zone
155,132,1,2025-07-01T00:39:14,2025-07-01T00:55:21,1,11.8,1,N,1,44.3,1.0,0.5,14.05,0.0,1.0,60.85,0.0,0.0,0.0,2025,7,1,0,16.12,Queens,JFK Airport,Airports,Brooklyn,Marine Park/Mill Basin,Boro Zone
263,79,2,2025-07-01T00:15:26,2025-07-01T00:29:39,1,3.87,1,N,1,17.7,1.0,0.5,4.69,0.0,1.0,28.14,2.5,0.0,0.75,2025,7,1,0,14.22,Manhattan,East Village,Yellow Zone,Manhattan,Yorkville West,Yellow Zone
262,140,2,2025-07-01T00:40:58,2025-07-01T00:44:15,1,0.85,1,N,1,5.8,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,0.0,2025,7,1,0,3.28,Manhattan,Lenox Hill East,Yellow Zone,Manhattan,Yorkville East,Yellow Zone
50,114,2,2025-07-01T00:28:12,2025-07-01T00:39:49,2,2.54,1,N,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75,2025,7,1,0,11.62,Manhattan,Greenwich Village South,Yellow Zone,Manhattan,Clinton West,Yellow Zone
197,132,2,2025-07-01T00:38:17,2025-07-01T00:55:44,1,6.37,1,N,1,26.8,1.0,0.5,5.86,0.0,1.0,36.91,0.0,1.75,0.0,2025,7,1,0,17.45,Queens,JFK Airport,Airports,Queens,Richmond Hill,Boro Zone


In [0]:
silver_df.printSchema()

print("Final Silver Rows :", silver_df.count())

root
 |-- DOLocationID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (null